In [1]:
#index ek data structure hai jo datbase ko batati hai ki kisi column ki values kaha  "sorted order" me rakhi hai - bina index ke , database ko har row check karni padti hai (Full table scan), jo bade dataset par slow hota hai. Index se database seedha relevant rows tak pahuch jata hai.

In [2]:
#01:pehle bina index ke query ka time measure karo (baseline):

In [3]:
import pandas as pd
import sqlite3
import time

conn = sqlite3.connect('../data/db/ecommerce.db')

start = time.time()
q1 = pd.read_sql("""
    SELECT * FROM transactions WHERE "Customer ID" = 14646
""", conn)
end = time.time()
print(f"Without index: {end - start:.4f} seconds")
print(q1.shape)

Without index: 0.5141 seconds
(3890, 8)


In [5]:
#02:EXPLAIN QUERY PLAN se dekho database internally kya kar raha hai:

In [6]:
plan = pd.read_sql("""
    EXPLAIN QUERY PLAN
    SELECT * FROM transactions WHERE "Customer ID" = 14646
""", conn)
print(plan)

   id  parent  notused             detail
0   2       0      216  SCAN transactions


In [7]:
#agar output me SCAN transactions dikhe (SEARCH nahi), matlab puri table scan ho rahi hai - index nahi use ho raha.

In [8]:
#03:Ab index banao us column par ji per baar-baar filter/join karte ho:

In [9]:
cursor = conn.cursor()
cursor.execute('CREATE INDEX IF NOT EXISTS idx_customer_id ON transactions("Customer ID")')
conn.commit()
print("Index created")

Index created


In [10]:
#04:same query dobara chalao or time compare karo:

In [11]:
start = time.time()
q2 = pd.read_sql("""
    SELECT * FROM transactions WHERE "Customer ID" = 14646
""", conn)
end = time.time()
print(f"With index: {end - start:.4f} seconds")

With index: 0.0227 seconds


In [12]:
#05:EXPLAIN QUERY PLAN dobara check karo - ab SEARCH dikhna chaiye, SCAN nahi:

In [13]:
plan2 = pd.read_sql("""
    EXPLAIN QUERY PLAN
    SELECT * FROM transactions WHERE "Customer ID" = 14646
""", conn)
print(plan2)

   id  parent  notused                                             detail
0   3       0       61  SEARCH transactions USING INDEX idx_customer_i...


In [14]:
#06:or bhi useful indexes banao (jo baar-baar JOIN/WHERE/GROUP BY me use hue hai):

In [15]:
cursor.execute('CREATE INDEX IF NOT EXISTS idx_invoice ON transactions(Invoice)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_country ON transactions(Country)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_invoicedate ON transactions(InvoiceDate)')
conn.commit()
print("Additional indexes created")

Additional indexes created


In [16]:
#07: JOIN query par bhi test karo (orders/customers tables per bhi index lagao):

In [17]:
cursor.execute('CREATE INDEX IF NOT EXISTS idx_orders_customerid ON orders("Customer ID")')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_customers_customerid ON customers("Customer ID")')
conn.commit()

start = time.time()
q3 = pd.read_sql("""
    SELECT c."Customer ID", c.Country, SUM(o.OrderValue) as total_spend
    FROM customers c
    INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
    GROUP BY c."Customer ID", c.Country
""", conn)
end = time.time()
print(f"JOIN query time: {end - start:.4f} seconds")

JOIN query time: 0.1470 seconds


In [18]:
#important note = Index har column par lagana thik nahi hai - har index write speed(INSERT/UPDATE) ko slow karta hai aur disk space leta hai. Index sirf un columns par lagao jo baar-baar WHERE, JOIN ON, ya ORDER BY me use hote hai.

In [19]:
#practice questions

In [20]:
#1.EXPLAIN QUERY PLAN se check karo Day 12 wala "top customers per country" query index lagne se pehle aur baad mein kaisa perform karta hai.

In [21]:
#

In [22]:
#2.sqlite_master table query karke dekho abhi tak kitne indexes ban chuke hain.

In [23]:
#

In [24]:
indexes = pd.read_sql("SELECT name, tbl_name FROM sqlite_master WHERE type='index'", conn)
print(indexes)

                       name      tbl_name
0           idx_customer_id  transactions
1               idx_invoice  transactions
2               idx_country  transactions
3           idx_invoicedate  transactions
4     idx_orders_customerid        orders
5  idx_customers_customerid     customers


In [25]:
conn.close()